# Phase II_02: U-Net Training on Change Detection

**Objective**: Train U-Net to classify burn severity from spectral change

**Key innovation**: 8-channel input (pre + post RGBN) with z-score normalization

**Input**: Pre-fire and post-fire RGBN images (concatenated to 8 channels)

**Output**: 7-class burn severity labels

## Setup: Mount Drive and Load Dependencies

In [ ]:
%pip install -q torch torchvision numpy matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Setup complete")
print("Note: This notebook loads pre-fire and post-fire RGBN images from Phase II_01")

In [ ]:
class ChangeDetectionDataset(Dataset):
    """Dataset that loads pre and post RGBN images and concatenates them.

    Input: 8 channels (pre_rgbn + post_rgbn, each [4, 512, 512])
    Output: 7-class burn severity labels
    Features:
    - Data augmentation (flip, rotate, zoom/crop) for training split only
    - Proper label alignment: uses indices [N + idx] for post-fire labels
    - Normalization is applied in training loop (not here) to avoid device conflicts
    """

    def __init__(self, pre_rgbn_tensor, post_rgbn_tensor, labels_tensor, split='train',
                 fold_path=None, augment=False):
        """
        Args:
            pre_rgbn_tensor: [N, 4, 512, 512] pre-fire RGBN images
            post_rgbn_tensor: [N, 4, 512, 512] post-fire RGBN images
            labels_tensor: [2N, 512, 512] severity labels (N pre-fire, N post-fire)
            split: 'train', 'val', or 'test'
            fold_path: Optional path to metadata JSON for fold information
            augment: Whether to apply data augmentation (only for train split)
        """
        self.pre_rgbn = pre_rgbn_tensor
        self.post_rgbn = post_rgbn_tensor
        self.labels = labels_tensor
        self.split = split
        self.augment = augment

        # Verify tensor shapes
        assert self.pre_rgbn.shape[0] == self.post_rgbn.shape[0], \
            f"Pre and post must have same N: {self.pre_rgbn.shape[0]} vs {self.post_rgbn.shape[0]}"
        assert self.labels.shape[0] == 2 * self.pre_rgbn.shape[0], \
            f"Labels must be 2N: got {self.labels.shape[0]}, expected {2 * self.pre_rgbn.shape[0]}"

        num_samples = self.pre_rgbn.shape[0]

        # If fold information available, filter by split
        if fold_path and Path(fold_path).exists():
            with open(fold_path, 'r') as f:
                metadata = json.load(f)

            # Build fold mapping from metadata
            self.fold_map = {}
            sample_idx = 0
            for fire in metadata['fires']:
                for sample in fire['samples']:
                    fold_name = sample['fold']
                    self.fold_map[sample_idx] = fold_name
                    sample_idx += 1

            # Filter indices by split
            fold_to_split = {'val': 'val', 'train': 'train', 'test': 'test'}
            self.indices = [i for i in range(num_samples)
                           if fold_to_split.get(self.fold_map.get(i), 'unknown') == split]
        else:
            # If no fold info, use all samples (warning: this mixes train/val/test)
            self.indices = list(range(num_samples))
            print(f"⚠ Warning: No fold information provided. Using all {len(self.indices)} samples for split '{split}'")

        print(f"✓ Dataset '{split}': {len(self.indices)} samples, augment={augment}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        sample_idx = self.indices[idx]

        # Load pre and post images, concatenate to 8 channels
        pre_img = self.pre_rgbn[sample_idx].float()  # [4, 512, 512]
        post_img = self.post_rgbn[sample_idx].float()  # [4, 512, 512]
        image = torch.cat([pre_img, post_img], dim=0)  # [8, 512, 512]

        # Get post-fire label (at index N + idx in labels tensor)
        label = self.labels[len(self.pre_rgbn) + sample_idx].long()  # [512, 512]

        # Apply data augmentation (only for training split)
        if self.augment:
            image, label = self._augment(image, label)

        return image, label

    def _augment(self, image, label):
        """Apply random augmentations to image and label."""
        # Random horizontal flip
        if torch.rand(1).item() > 0.5:
            image = torch.flip(image, dims=[2])
            label = torch.flip(label, dims=[1])

        # Random vertical flip
        if torch.rand(1).item() > 0.5:
            image = torch.flip(image, dims=[1])
            label = torch.flip(label, dims=[0])

        # Random rotation (90, 180, 270 degrees)
        k = torch.randint(0, 4, (1,)).item()
        if k > 0:
            image = torch.rot90(image, k=k, dims=[1, 2])
            label = torch.rot90(label, k=k, dims=[0, 1])

        # Random zoom/crop: crop to random region then interpolate back
        if torch.rand(1).item() > 0.5:
            crop_size = torch.randint(384, 512, (1,)).item()  # Random crop size 384-512
            h_start = torch.randint(0, 512 - crop_size, (1,)).item()
            w_start = torch.randint(0, 512 - crop_size, (1,)).item()

            image = image[:, h_start:h_start+crop_size, w_start:w_start+crop_size]
            label = label[h_start:h_start+crop_size, w_start:w_start+crop_size]

            # Interpolate back to 512x512
            image = F.interpolate(image.unsqueeze(0), size=(512, 512), mode='bilinear', align_corners=False).squeeze(0)
            label = F.interpolate(label.unsqueeze(0).unsqueeze(0).float(), size=(512, 512), mode='nearest').squeeze(0).squeeze(0).long()

        return image, label

# Load Phase II_01 outputs from Drive
print(f"Loading Phase II_01 outputs from Drive...\n")

labels_dir = Path('/content/drive/MyDrive/RETINNA_cache/phase2/II_01_relabeling')
if not labels_dir.exists():
    raise FileNotFoundError(f"Phase II_01 outputs not found at {labels_dir}")

# Find latest files
label_files = sorted(labels_dir.glob('multi_class_labels_*.pt'))
pre_rgbn_files = sorted(labels_dir.glob('pre_rgbn_*.pt'))
post_rgbn_files = sorted(labels_dir.glob('post_rgbn_*.pt'))

if not label_files or not pre_rgbn_files or not post_rgbn_files:
    raise FileNotFoundError(f"Missing Phase II_01 outputs in {labels_dir}")

latest_labels_file = label_files[-1]
latest_pre_rgbn_file = pre_rgbn_files[-1]
latest_post_rgbn_file = post_rgbn_files[-1]

print(f"✓ Loading labels: {latest_labels_file.name}")
labels_tensor = torch.load(latest_labels_file)
print(f"  Shape: {labels_tensor.shape}")

print(f"✓ Loading pre-fire RGBN: {latest_pre_rgbn_file.name}")
pre_rgbn_tensor = torch.load(latest_pre_rgbn_file)
print(f"  Shape: {pre_rgbn_tensor.shape}")

print(f"✓ Loading post-fire RGBN: {latest_post_rgbn_file.name}")
post_rgbn_tensor = torch.load(latest_post_rgbn_file)
print(f"  Shape: {post_rgbn_tensor.shape}")

# Try to load metadata for fold information (optional)
metadata_paths = [
    Path('/content/drive/MyDrive/RETINNA_DATA/metadata/cabuaur_metadata.json'),
    Path('/content/RETINNA/docs/cabuaur_metadata.json'),
]
metadata_path = None
for path in metadata_paths:
    if path.exists():
        metadata_path = path
        print(f"\n✓ Found metadata at {path}")
        break

if not metadata_path:
    print(f"\n⚠ Metadata not found. Will use all {len(pre_rgbn_tensor)} samples (train/val mixed)")

print(f"\n✓ All Phase II_01 outputs loaded successfully")

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels // 2, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Handle size mismatch from skip connection
        if x2.size() != x1.size():
            x1 = F.pad(x1, (0, x2.size(3) - x1.size(3), 0, x2.size(2) - x1.size(2)))
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    """U-Net for 7-class burn severity segmentation from 8-channel input (pre + post RGBN)."""
    def __init__(self, in_channels=8, out_channels=7, bilinear=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.bilinear = bilinear
        
        factor = 2 if bilinear else 1
        
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        x = self.outc(x)
        return x

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_channels=8, out_channels=7, bilinear=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: U-Net")
print(f"  Input channels: 8 (pre-fire + post-fire RGBN)")
print(f"  Output classes: 7 (burn severity)")
print(f"  Total parameters: {total_params:,}")
print(f"  Device: {device}")

In [ ]:
# Compute normalization statistics from training set only (no val/test leakage)
print("Computing channel normalization statistics from training data...\n")

# Temporary training dataset without augmentation to compute mean/std
train_dataset_temp = ChangeDetectionDataset(
    pre_rgbn_tensor, post_rgbn_tensor, labels_tensor,
    split='train', fold_path=metadata_path,
    augment=False
)

# Compute mean and std for all 8 channels from training data only
channel_means = []
channel_stds = []

for idx in range(len(train_dataset_temp)):
    sample_idx = train_dataset_temp.indices[idx]
    pre_img = pre_rgbn_tensor[sample_idx].float()
    post_img = post_rgbn_tensor[sample_idx].float()
    image = torch.cat([pre_img, post_img], dim=0)  # [8, 512, 512]
    
    for c in range(8):
        channel_means.append(image[c].mean().item())
        channel_stds.append(image[c].std().item())

# Compute per-channel mean and std
channel_means = torch.tensor(channel_means).view(8, -1).mean(dim=1)
channel_stds = torch.tensor(channel_stds).view(8, -1).mean(dim=1)

print(f"Channel normalization statistics (from training set):")
for c in range(8):
    channel_name = f"pre_{['R', 'G', 'B', 'N'][c % 4]}" if c < 4 else f"post_{['R', 'G', 'B', 'N'][c % 4]}"
    print(f"  Channel {c} ({channel_name}): mean={channel_means[c]:.4f}, std={channel_stds[c]:.4f}")

print(f"\n✓ Normalization statistics computed (will apply in training loop, not dataset)")

In [ ]:
# Create final datasets with augmentation (no normalization in dataset)
print("Creating training and validation datasets...\n")

train_dataset = ChangeDetectionDataset(
    pre_rgbn_tensor, post_rgbn_tensor, labels_tensor,
    split='train', fold_path=metadata_path,
    augment=True  # Enable augmentation for training
)

val_dataset = ChangeDetectionDataset(
    pre_rgbn_tensor, post_rgbn_tensor, labels_tensor,
    split='val', fold_path=metadata_path,
    augment=False  # No augmentation for validation
)

# Compute class weights from training label distribution (no val/test leakage)
print("Computing class weights from training label distribution...\n")

class_counts = torch.zeros(7)
for idx in range(len(train_dataset)):
    sample_idx = train_dataset.indices[idx]
    label = labels_tensor[len(pre_rgbn_tensor) + sample_idx].long()
    for class_id in range(7):
        class_counts[class_id] += (label == class_id).sum().item()

# Compute class weights: inverse frequency with sqrt dampening
# sqrt dampening reduces weight spread to prevent loss instability
# Alternative: 1.0 / (class_counts + 1.0) produces 860:1 ratio (too extreme)
# This gives ~30:1 ratio which is balanced by WeightedRandomSampler
class_weights = 1.0 / torch.sqrt(class_counts + 1.0)
class_weights = class_weights / class_weights.mean()

print(f"Class distribution (training set):")
for c in range(7):
    count = class_counts[c].item()
    percent = 100 * count / class_counts.sum().item()
    weight = class_weights[c].item()
    print(f"  Class {c}: {count:,} pixels ({percent:.2f}%) -> weight={weight:.4f}")

print(f"\n✓ Datasets and class weights ready")

## Training Configuration

In [ ]:
# ============================================================================
# IMPORT CONFIGURABLE LOSS FUNCTIONS AND CONFIG
# ============================================================================
import sys
import os

# Add current notebook directory to path (works in both local and Colab)
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '/content/RETINNA/notebooks'
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from losses import get_loss
from config import LOSS_CONFIG

# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

batch_size = 4
learning_rate = 1e-3
num_epochs = 20
weight_decay = 1e-5

# ============================================================================
# CREATE LOSS FUNCTION (CONFIGURABLE AT RUNTIME VIA config.py)
# ============================================================================

print(f"Loss Function Configuration:")
print(f"  Name: {LOSS_CONFIG['name']}")
print(f"  Params: {LOSS_CONFIG['params']}")
print(f"  Use class weights: {LOSS_CONFIG['use_class_weights']}")

# Apply class weights to loss function if configured
loss_class_weights = class_weights if LOSS_CONFIG['use_class_weights'] else None

# Create loss function with params from config
criterion = get_loss(
    LOSS_CONFIG['name'],
    class_weights=loss_class_weights,
    **LOSS_CONFIG['params']
)
criterion = criterion.to(device)

print(f"\n✓ Loss function created: {LOSS_CONFIG['name'].upper()}")

# ============================================================================
# DATA LOADERS & OPTIMIZER
# ============================================================================

print("Creating data loaders...\n")
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-7
)

print(f"Training configuration:")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")
print(f"  Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")
print(f"  Epochs: {num_epochs}")
print(f"  Optimizer: Adam")
print(f"\nData loaders:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Device: {device}")
print(f"\n" + "="*70)
print(f"To test a different loss function:")
print(f"1. Edit config.py and change LOSS_CONFIG['name']")
print(f"2. Restart the kernel")
print(f"3. Re-run this cell and the training loop")
print(f"="*70)

## Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device, channel_means, channel_stds):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    
    # Move normalization stats to device
    channel_means = channel_means.to(device)
    channel_stds = channel_stds.to(device)
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        # Move batch to device FIRST
        images = images.to(device)
        labels = labels.to(device)
        
        # THEN apply z-score normalization (now all on same device)
        mean = channel_means.view(8, 1, 1)
        std = channel_stds.view(8, 1, 1)
        images = (images - mean) / (std + 1e-8)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        total_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_pixels += labels.numel()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_pixels
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device, channel_means, channel_stds):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    
    # Move normalization stats to device
    channel_means = channel_means.to(device)
    channel_stds = channel_stds.to(device)
    
    with torch.no_grad():
        for images, labels in val_loader:
            # Move batch to device FIRST
            images = images.to(device)
            labels = labels.to(device)
            
            # THEN apply z-score normalization
            mean = channel_means.view(8, 1, 1)
            std = channel_stds.view(8, 1, 1)
            images = (images - mean) / (std + 1e-8)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            predictions = outputs.argmax(dim=1)
            total_correct += (predictions == labels).sum().item()
            total_pixels += labels.numel()
    
    avg_loss = total_loss / len(val_loader)
    accuracy = total_correct / total_pixels
    return avg_loss, accuracy

# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, channel_means, channel_stds)
    val_loss, val_acc = validate(model, val_loader, criterion, device, channel_means, channel_stds)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    
    # Step scheduler based on validation loss
    scheduler.step(val_loss)

## Visualization and Metrics

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid()

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

print(f"✓ Training history saved")

## Save Model

In [ ]:
from datetime import datetime

# Save model
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_save_path = f'/content/drive/MyDrive/RETINNA_cache/phase2/II_02_training/unet_model_{timestamp}.pt'

# Create directory if it doesn't exist
model_dir = Path(model_save_path).parent
model_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'epoch': num_epochs,
    'config': {
        'in_channels': 8,
        'out_channels': 7,
        'input_type': '8-channel (pre-fire + post-fire RGBN)',
        'label_source': 'Phase II_01 RdNBR classification',
    },
    'normalization': {
        'channel_mean': channel_means.cpu(),
        'channel_std': channel_stds.cpu(),
    },
    'class_weights': class_weights.cpu(),
    'history': history,
}, model_save_path)

print(f"✓ Model saved: {model_save_path}")
print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
print(f"  Final val loss: {history['val_loss'][-1]:.4f}")
print(f"  Final train acc: {history['train_acc'][-1]:.4f}")
print(f"  Final val acc: {history['val_acc'][-1]:.4f}")
print(f"\n✓ Checkpoint includes:")
print(f"  - model_state_dict")
print(f"  - normalization (channel_mean, channel_std)")
print(f"  - class_weights")
print(f"  - training history")

## Summary

**Phase II_02 Complete: U-Net Training on 8-Channel Change Detection**

**Model specifications:**
- Input: 8 channels (pre-fire RGBN [0:4] + post-fire RGBN [4:8])
- Output: 7-class burn severity segmentation
- Architecture: U-Net with bilinear interpolation

**Data processing:**
- Z-score normalization: computed from training set only (no val/test leakage)
- Data augmentation: flip, rotate, zoom/crop (training split only)
- Label alignment: post-fire labels at indices [N + idx]
- Class weights: computed from training distribution (1 / frequency, normalized)

**Checkpoint contents:**
- model_state_dict: trained weights
- normalization: channel_mean and channel_std for 8 channels
- class_weights: 7-class weights from training distribution
- history: train/val loss and accuracy per epoch
- config: model specifications and input type

**Ready for Phase III:**
- Model accepts 8-channel inputs (pre + post)
- Can directly transfer to NAIP temporal pairs
- Checkpoint includes all necessary normalization statistics

**Next steps:**
1. Acquire NAIP temporal pairs (pre/post fire)
2. Load pre/post bands separately
3. Concatenate to 8 channels
4. Apply stored normalization statistics
5. Run inference
6. Validate results